# repairs

> Apply corrections a person makes from the scan, number verses the model left unnumbered, mark chapter breaks, drop pages that are not the text, and tidy sūtras

In [ ]:
#| default_exp repairs

In [ ]:
#| hide
from fastcore.test import *

In [ ]:
#| export
import re
from difflib import SequenceMatcher
from lekhaka.verses import vnum, blocks, kind, _bk

Some mistakes in a reading cannot be fixed by rule. A person has to look at the scan, decide what the page really says, and write that decision down. The functions here apply those decisions, along with a few repairs that need a hint from you, such as where a chapter starts.

Most of them work on pages of blocks, `[{page, blocks}]` (or `items`), before `assemble` puts the verses into chapters. A block is one piece of a page as the vision model read it: a verse, a heading, a colophon (the line that closes a chapter), a commentary or prose. These functions change the pages in place and also return them. `fix_chapters` and `sutras` work later, on the chapters `assemble` returns. Every change goes into `probs`, a list you pass in, so the list of problems also says what was changed and why.

They were written for books vedicreader read and had to repair again and again: the fix tables for the Bṛhat Saṃhitā, the Muhūrta Cintāmaṇi and the Nārada Saṃhitā, the unnumbered runs of the Phaladīpikā, the second readings of the Jātaka Pārijāta, and the chapter plans of the Grahacintāmaṇi.

## A table of fixes

Each correction is a fix row, `(page, item, kind, read, printed, why)`. `page` and `item` say where the mistake is: the page's name and the block's position on that page, counting from 0. `read` is the text as the model read it, `printed` is what the page prints, and `why` is your reason, which goes into `probs` with the change. `kind` says what to do:

- `fix` replaces the first occurrence of `read` in the block's text with `printed`, and `fix_all` replaces every occurrence.
- `doubt` is a fix you are not sure of. It is applied only when `witness`, a dict keyed by `(page, item, read)`, holds `'agrees'` for it, meaning another edition reads the same way. Otherwise the text is kept and the doubt is recorded.
- `slip` records a mistake that the printed book itself makes. The text is kept, since it matches the print.
- `number` sets the block's number to `printed`.
- `drop` removes the block.
- `missing` adds a block the model skipped. `printed` is the whole block as a dict, and it goes in after position `item`, or first on the page when `item` is -1.

Blocks are removed and added only after every row has been read, so the positions in all the rows refer to the page as the model read it. A row for a page that is not in the reading is ignored. A row whose position is past the end of the page, or whose `read` text is not in the block, is not applied, and that is recorded in `probs`. The test below applies rows of several kinds to one small page.

In [ ]:
#| export
def apply_fixes(pages, rows, witness:dict=None, probs:list=None) -> list:
    "Apply a table of fix rows `(page, item, kind, read, printed, why)` to pages of blocks, recording each change in `probs`."
    probs = [] if probs is None else probs
    by, adds = {p['page']: p for p in pages}, []
    for pg, i, kind, a, b, why in rows:
        if (p := by.get(pg)) is None: continue
        bl = blocks(p)
        if kind == 'missing': adds.append((pg, i, b, why)); continue
        it = bl[i] if isinstance(i, int) and 0 <= i < len(bl) else None
        if it is None: probs.append(f'{pg}: block {i} is gone from the reading; {kind} {a} → {b} not applied'); continue
        if kind == 'drop': it['drop'] = True; probs.append(f"{pg}: `{(it.get('text') or '')[:30]}` dropped ({why})"); continue
        if kind == 'number': it['n'] = b; probs.append(f'{pg}: verse number read {a}, printed {b}'); continue
        if a not in (it.get('text') or ''): probs.append(f'{pg}: `{a}` not in block {i}; the {kind} to `{b}` not applied'); continue
        w = (witness or {}).get((pg, i, a))
        if kind in ('fix', 'fix_all') or (kind == 'doubt' and w == 'agrees'):
            m = it['text'].count(a) if kind == 'fix_all' else 1
            it['text'] = it['text'].replace(a, b, m)
            probs.append(f'{pg}: read `{a}`, printed `{b}`' + (f', {m} times' if m > 1 else '') + (f' ({why})' if why else '') + (' [a doubt the witness settles]' if kind == 'doubt' else ''))
        elif kind == 'doubt': probs.append(f'{pg}: doubt: `{a}` kept; perhaps `{b}` ({why})')
        elif kind == 'slip': probs.append(f"{pg}: the print's slip, kept: `{a}` ({why})")
    for _, (pg, after, blk, why) in sorted(enumerate(adds), key=lambda x: (-(x[1][1] if isinstance(x[1][1], int) else 999), -x[0])):
        blocks(by[pg]).insert(after + 1 if isinstance(after, int) and after >= 0 else 0, dict(blk, added=True))
        probs.append(f"{pg}: {blk.get('kind')} {blk.get('n') or ''} missing from the reading, put in: {(blk.get('text') or '')[:40]}")
    for p in pages: p[_bk(p)] = [b for b in blocks(p) if not b.get('drop')]
    return pages

In [ ]:
_pg = lambda: [dict(page='p1', blocks=[dict(kind='verse', n=1, text='रामः वनं गच्छति ॥१॥'), dict(kind='verse', n=3, text='सीता ॥३॥'), dict(kind='prose', text='a note')])]
_p = []
_r = apply_fixes(_pg(), [('p1', 0, 'fix', 'गच्छति', 'गच्छत्', 'the print'), ('p1', 1, 'number', 3, 2, ''), ('p1', 2, 'drop', None, None, 'a footnote'),
                         ('p1', 1, 'missing', None, dict(kind='verse', n=3, text='लक्ष्मणः ॥३॥'), 'at the foot'), ('p1', 0, 'doubt', 'रामः', 'राम', 'faint')], probs=_p)
test_eq([(b['n'], b['text']) for b in _r[0]['blocks']], [(1, 'रामः वनं गच्छत् ॥१॥'), (2, 'सीता ॥३॥'), (3, 'लक्ष्मणः ॥३॥')])
test_eq(len(_p), 5); assert any('doubt' in x for x in _p)
test_eq(apply_fixes(_pg(), [('p1', 0, 'doubt', 'रामः', 'राम', '')], witness={('p1', 0, 'रामः'): 'agrees'})[0]['blocks'][0]['text'][:3], 'राम')
_p = []; test_eq(apply_fixes([dict(page=0, blocks=[dict(kind='verse', text='क क ख')])], [(0, 0, 'fix_all', 'क', 'ग', '')], probs=_p)[0]['blocks'][0]['text'], 'ग ग ख')
test_eq(_p, ['0: read `क`, printed `ग`, 2 times'])

## Numbers

Models often miss a verse number, especially one printed small or faint, or at the edge of the page. Two functions recover such numbers without asking a person.

If verse 4 follows verse 1 with two unnumbered verse blocks between them, those two must be verses 2 and 3. `number_runs` numbers a run of unnumbered verses when it exactly fills the gap between two numbered ones, and leaves it alone otherwise. Only blocks of kind `verse` count. The count starts again from 0 after a colophon, and after a heading that matches `reset` (ध्याय by default, as in अध्यायः), so the first verses of a new chapter can be numbered from 1.

`borrow_numbers` uses a second reading of the same pages, by another model or another run. For each unnumbered verse, it finds the most similar numbered verse on the same page of the other reading and takes its number, if the two texts are at least `at` alike (0.6 by default, where 1 means identical) and no verse on this page already has that number.

In [ ]:
#| export
def number_runs(pages, probs:list=None, reset:str=r'ध्याय') -> list:
    "Number each run of unnumbered verses that exactly fills the gap between two numbered ones."
    probs = [] if probs is None else probs
    a, run = 0, []
    for p in pages:
        for b in blocks(p):
            k = kind(b)
            if k == 'colophon' or (k == 'heading' and reset and re.search(reset, b.get('text') or '')): a, run = 0, []; continue
            if k != 'verse': continue
            if (n := vnum(b.get('n'))) is None: run.append((p['page'], b)); continue
            if run and n - a - 1 == len(run):
                for j, (pg, x) in enumerate(run, 1): x['n'] = a + j; probs.append(f'{pg}: verse printed without its number taken as {a + j}, between {a} and {n}')
            a, run = n, []
    return pages

def borrow_numbers(pages, other, probs:list=None, at:float=.6) -> list:
    "Give each unnumbered verse the number of the most similar verse on the same page of `other`, a second reading of the same pages."
    probs = [] if probs is None else probs
    theirs = {p['page']: [(n, b.get('text') or '') for b in blocks(p) if kind(b) == 'verse' and (n := vnum(b.get('n'))) is not None] for p in other}
    for p in pages:
        if not (th := theirs.get(p['page'])): continue
        have = {vnum(b.get('n')) for b in blocks(p) if kind(b) == 'verse'}
        for b in blocks(p):
            if kind(b) != 'verse' or b.get('n') is not None: continue
            sc, n = max((SequenceMatcher(None, b.get('text') or '', t).ratio(), n) for n, t in th)
            if sc >= at and n not in have: b['n'] = n; have.add(n); probs.append(f"{p['page']}: verse {n}'s number, lost in this reading, taken from the other ({sc:.2f})")
    return pages

In [ ]:
_ps = [dict(page=0, blocks=[dict(kind='verse', n=1, text='a'), dict(kind='verse', text='b'), dict(kind='verse', text='c'), dict(kind='verse', n=4, text='d'),
                             dict(kind='colophon', text='इति'), dict(kind='verse', text='e'), dict(kind='verse', n=3, text='f')])]
test_eq([b.get('n') for b in number_runs(_ps)[0]['blocks']], [1, 2, 3, 4, None, None, 3])   # after the colophon one verse cannot fill 1..2
_mine = [dict(page=0, blocks=[dict(kind='verse', text='धर्मक्षेत्रे कुरुक्षेत्रे'), dict(kind='verse', n=2, text='दृष्ट्वा तु')])]
_theirs = [dict(page=0, blocks=[dict(kind='verse', n='१', text='धर्मक्षेत्रे कुरुक्षेत्रे समवेता'), dict(kind='verse', n='२', text='दृष्ट्वा तु')])]
test_eq([b['n'] for b in borrow_numbers(_mine, _theirs)[0]['blocks']], [1, 2])

In [ ]:
_it = [dict(page=0, items=[dict(kind='verse', n=1, text='a'), dict(kind='verse', text='b'), dict(kind='head', text='द्वितीयोऽध्यायः'), dict(kind='verse', text='c'), dict(kind='verse', n=2, text='d')])]
test_eq([b.get('n') for b in number_runs(_it)[0]['items']], [1, None, None, 1, 2])   # `items` read as blocks, a `head` as a heading
test_eq([b['n'] for b in borrow_numbers([dict(page=0, items=[dict(kind='verse', text='धर्मक्षेत्रे कुरुक्षेत्रे')])], _theirs)[0]['items']], [1])
test_eq(len(apply_fixes([dict(page=0, items=[dict(kind='verse', text='x')])], [(0, 0, 'drop', None, None, '')])[0]['items']), 0)

## Chapters by plan

Some books start a chapter in a way `assemble` cannot see: with a heading the model did not mark as one, or with no mark at all because a leaf is missing. `cut_by_plan` takes a plan, a list of `(page, item, why)` rows saying where each such chapter begins. `item` is either a block's position on the page or the first words of a block's text, usually a heading, and `why` is optional. Before that block it puts a block of kind `start`, and `assemble` begins a new chapter at a `start` block. Each cut is recorded in `probs`, and so is a row whose page or block cannot be found.

In [ ]:
#| export
def cut_by_plan(pages, plan, probs:list=None) -> list:
    "Insert a chapter-opening `start` block at each `(page, item, why)` row of `plan`, where `item` is a block's position or the first words of its text."
    probs = [] if probs is None else probs
    by = {p['page']: p for p in pages}
    for pg, item, *why in plan:
        if (p := by.get(pg)) is None: probs.append(f'{pg}: the plan names a page the reading lacks'); continue
        bl = blocks(p)
        i = item if isinstance(item, int) else next((j for j, b in enumerate(bl) if (b.get('text') or '').startswith(item)), None)
        if i is None or not 0 <= i < len(bl): probs.append(f'{pg}: the plan\'s chapter start {item!r} is not on the page'); continue
        b, y = bl[i], f' ({why[0]})' if why and why[0] else ''
        w = f'verse {m}' if kind(b) == 'verse' and (m := vnum(b.get('n'))) is not None else f"`{(b.get('text') or '')[:30]}`"
        bl.insert(i, dict(kind='start')); probs.append(f'{pg}: a chapter starts at {w}, by the plan{y}')
    return pages

In [ ]:
from lekhaka.verses import assemble
_pl = [dict(page=0, blocks=[dict(kind='heading', text='கடவுள் வணக்கம்'), dict(kind='verse', n=1, text='x'), dict(kind='heading', text='வாரநயம்'), dict(kind='verse', n=2, text='y')])]
_p = []; _c, _ = assemble(cut_by_plan(_pl, [(0, 'வாரநயம்')], _p)); test_eq(_p, ['0: a chapter starts at `வாரநயம்`, by the plan'])
test_eq([(c['heading'], c['colophon'], [v['n'] for v in c['verses']]) for c in _c], [('கடவுள் வணக்கம்', None, [1]), ('வாரநயம்', None, [2])])
_p = []; cut_by_plan(_pl, [(5, 0), (0, 'அது')], _p); test_eq(len(_p), 2)
_p = []; cut_by_plan([dict(page='p2', items=[dict(kind='verse', n='२', text='x')])], [('p2', 0, 'the leaf before it lost')], _p)
test_eq(_p, ['p2: a chapter starts at verse 2, by the plan (the leaf before it lost)'])

## Pages that are not the text

Printed books often carry pages whose verses are not part of the work, such as a preface, an index, or an appendix quoting another work. The Phaladīpikā edition, for example, prints verses of the Jātakābharaṇa after verse 26.48. `drop_pages` takes a list of `(first, last)` page ranges, both ends included, and removes the verse blocks on those pages. Headings and other blocks stay.

In [ ]:
#| export
def drop_pages(pages, ranges, probs:list=None) -> list:
    "Remove the verse blocks on every page in each `(first, last)` range, for pages that are not part of the text."
    probs = [] if probs is None else probs
    for a, b in ranges:
        for p in pages:
            if a <= p['page'] <= b and any(kind(x) == 'verse' for x in blocks(p)):
                p[_bk(p)] = [x for x in blocks(p) if kind(x) != 'verse']; probs.append(f"{p['page']}: verses dropped, not the text")
    return pages

In [ ]:
_p = []
_d = drop_pages([dict(page=k, items=[dict(kind='heading', text='h'), dict(kind='verse', n=k, text='v')]) for k in range(1, 5)], [(2, 3)], _p)
test_eq([len(p['items']) for p in _d], [2, 1, 1, 2]); test_eq(_p, ['2: verses dropped, not the text', '3: verses dropped, not the text'])

## After assembly

Once `assemble` has built the chapters, it is easier to name a verse by its chapter and number than by its page and position. `fix_chapters` takes fix rows `(chapter, verse, kind, read, printed, why)`, where `verse` is the verse's number. `chapter` is the chapter's position counting from 1, unless you pass `key`, a function `key(i, chapter)` that names chapters some other way, such as the edition's own chapter numbers or a `(part, no)` pair. A `chapter` of None applies the row to every chapter that has that verse. The rows are applied in order, so after a `start` row has split a chapter, the rows that follow count the chapters as split.

The kinds are:

- `fix` and `fix_all` replace `read` with `printed` in the verse, once or every time, as in `apply_fixes`.
- `slip` and `doubt` are recorded and the text is kept. A `doubt` is never applied here, since `fix_chapters` takes no witness.
- `note` records `why` as a remark on the verse and changes nothing.
- `number` sets the verse's number to `printed`, and `drop` removes the verse.
- `missing` adds a verse the reading lacks, perhaps copied by hand from another copy of the book. `printed` is its text, or a dict such as `{text, page}`, and it goes in at its place by number. If the chapter already has a verse with that number, nothing is added.
- `recut` moves the boundary between a verse and the one before it, for a case `mend` cannot see, such as a four-line verse whose last line has no daṇḍa, so the model ran it into the next verse. The two verses are joined and split again so that the earlier one keeps the first `printed` lines.
- `start` begins a new chapter at the verse. This is for a lost leaf that took the end of one chapter and the start of the next, so the numbering does not restart where `assemble` would notice. The verse cannot be the first of its chapter. If a chapter holds two verses with that number, `printed` says which one: the page it is on, or 1 for the first and 2 for the second. The first part keeps the chapter's heading and the second keeps its colophon.

Any other kind raises a `ValueError`.

`sutras` is for works written in sūtras, short prose aphorisms, rather than verse. It puts each sūtra on one line, joining wrapped lines and removing a hyphen at a line break. Where a sūtra does not end in `close` (॥ by default), it removes a final । or | and adds `close`, recording that in `probs`.

In [ ]:
#| export
_KINDS = ('fix', 'fix_all', 'slip', 'doubt', 'note', 'number', 'drop', 'missing', 'recut', 'start')
def _at(vs, n): return next((i for i, v in enumerate(vs) if v['n'] == n), None)

def fix_chapters(chs, rows, probs:list=None, key=None) -> list:
    "Apply a table of fix rows `(chapter, verse, kind, read, printed, why)` to assembled chapters; a `chapter` of None means every chapter with that verse."
    probs, key = [] if probs is None else probs, key or (lambda i, c: i)
    for ch, n, k, a, b, why in rows:
        if k not in _KINDS: raise ValueError(f'{k}: not a kind of fix')
        w, y = f'{ch}.{n}' if ch is not None else f'{n}', f' ({why})' if why else ''
        cs = [c for i, c in enumerate(chs, 1) if ch is None or key(i, c) == ch]
        if k == 'note': probs.append(f'{w}: {why}'); continue
        if not cs: probs.append(f'{w}: no chapter {ch}; the {k} not applied'); continue
        if k == 'missing':
            vs = cs[0]['verses']
            if _at(vs, n) is not None: probs.append(f'{w}: verse {n} is in the reading; the missing verse not put in'); continue
            v = dict(n=n, text=b, page=None, heading=None) if isinstance(b, str) else dict(dict(page=None, heading=None), **b, n=n)
            vs.insert(next((i for i, x in enumerate(vs) if x['n'] > n), len(vs)), dict(v, added=True))
            probs.append(f"{w}: verse {n} missing from the reading, put in: {v['text'][:40]}{y}"); continue
        if k == 'start':
            hs = [(c, i) for c in cs for i, v in enumerate(c['verses']) if v['n'] == n and (not isinstance(b, str) or v['page'] == b)]
            hs = hs if isinstance(b, str) else hs[b - 1:b] if isinstance(b, int) else [(c, _at(c['verses'], n)) for c in cs]
            c, i = next(((c, i) for c, i in hs if i), (None, None))
            if c is None: probs.append(f'{w}: no verse {n} after the first of a chapter' + ('' if b is None else f' ({b})') + '; the start not applied'); continue
            vs, j = c['verses'], next(j for j, x in enumerate(chs) if x is c)
            chs[j:j + 1] = [dict(c, verses=vs[:i], colophon=None), dict(c, heading=None, verses=vs[i:])]
            probs.append(f"{vs[i]['page']}: a chapter starts at verse {w}{y}"); continue
        if k == 'recut':
            vs = cs[0]['verses']
            if not (i := _at(vs, n)): probs.append(f'{w}: no verse before {n} to recut it from'); continue
            ls = (vs[i - 1]['text'] + '\n' + vs[i]['text']).split('\n')
            vs[i - 1], vs[i] = dict(vs[i - 1], text='\n'.join(ls[:b])), dict(vs[i], text='\n'.join(ls[b:]))
            probs.append(f"{vs[i]['page']}: {w} begins at line {b + 1} of {n - 1}-{n} as read{y}"); continue
        hits = [(c, i) for c in cs if (i := _at(c['verses'], n)) is not None]
        if not hits: probs.append(f'{w}: verse {n} is gone from the reading; {k} {a} → {b} not applied'); continue
        for c, i in hits:
            v = c['verses'][i]; pg = f"{key(next(j for j, x in enumerate(chs, 1) if x is c), c)}.{n} ({v['page']})"
            if k == 'drop': del c['verses'][i]; probs.append(f"{pg}: `{v['text'][:30]}` dropped{y}")
            elif k == 'number': c['verses'][i] = dict(v, n=b); probs.append(f'{pg}: verse number read {a}, printed {b}')
            elif k == 'slip': probs.append(f"{pg}: the print's slip, kept: `{a}`{y}")
            elif k == 'doubt': probs.append(f'{pg}: doubt: `{a}` kept; perhaps `{b}`{y}')
            elif a not in v['text']: probs.append(f'{pg}: `{a}` not in the verse; the fix to `{b}` not applied')
            else:
                m = v['text'].count(a) if k == 'fix_all' else 1
                c['verses'][i] = dict(v, text=v['text'].replace(a, b, m)); probs.append(f'{pg}: read `{a}`, printed `{b}`' + (f', {m} times' if m > 1 else '') + y)
    return chs

def sutras(chs, probs:list=None, close:str='॥', key=None) -> list:
    "Put each sūtra on one line, rejoining words hyphenated at a line break, and end it with `close` where the reading lost it."
    probs, key = [] if probs is None else probs, key or (lambda i, c: i)
    for i, c in enumerate(chs, 1):
        out = []
        for v in c['verses']:
            u = re.sub(r'-\s*\n\s*', '', v['text']).replace('\n', ' ').strip()
            if not u.endswith(close): u = re.sub(r'[।|]$', '', u) + close; probs.append(f"{key(i, c)}.{v['n']} ({v['page']}): closing {close} supplied")
            out.append(dict(v, text=u))
        c['verses'] = out
    return chs

In [ ]:
V = lambda n, t, pg=1: dict(n=n, text=t, page=pg, heading=None)
_ch = lambda: [dict(heading='h1', colophon='c1', verses=[V(1, 'रामः गच्छति ॥१॥'), V(2, 'सीता ॥२॥'), V(4, 'क\nख\nग ॥४॥'), V(5, 'घ ॥५॥')]), dict(heading='h2', colophon='c2', verses=[V(1, 'रामः ॥१॥')])]
_p = []
_c = fix_chapters(_ch(), [(1, 1, 'fix', 'गच्छति', 'गच्छत्', 'typed'), (None, 1, 'fix', 'रामः', 'राम', ''), (1, 3, 'missing', None, 'लक्ष्मणः ॥३॥', 'from the 1905 print'),
                          (1, 5, 'recut', None, 2, ''), (1, 2, 'slip', 'सीता', None, 'so printed'), (2, 9, 'fix', 'x', 'y', ''), (1, 1, 'note', None, None, 'a lacuna'),
                          (1, 4, 'start', None, None, 'a leaf lost'), (2, 5, 'drop', None, None, 'a quotation')], _p)
test_eq([(c['heading'], c['colophon'], [(v['n'], v['text']) for v in c['verses']]) for c in _c],
        [('h1', None, [(1, 'राम गच्छत् ॥१॥'), (2, 'सीता ॥२॥'), (3, 'लक्ष्मणः ॥३॥')]), (None, 'c1', [(4, 'क\nख')]), ('h2', 'c2', [(1, 'राम ॥१॥')])])
test_eq(_p[:3], ['1.1 (1): read `गच्छति`, printed `गच्छत्` (typed)', '1.1 (1): read `रामः`, printed `राम`', '2.1 (1): read `रामः`, printed `राम`'])
test_eq(len(_p), 10); assert '2.9: verse 9 is gone from the reading; fix x → y not applied' in _p
test_eq([v['n'] for v in fix_chapters(_ch(), [((1, 'x'), 2, 'drop', None, None, '')], key=lambda i, c: (i, 'x'))[0]['verses']], [1, 4, 5])
test_fail(lambda: fix_chapters(_ch(), [(1, 1, 'mend', 'a', 'b', '')]))
_p = []; _c = fix_chapters([dict(heading=None, colophon=None, verses=[V(1, 'क क ॥१॥')])], [(1, 1, 'fix_all', 'क', 'ग', 'twice')], _p)
test_eq((_c[0]['verses'][0]['text'], _p), ('ग ग ॥१॥', ['1.1 (1): read `क`, printed `ग`, 2 times (twice)']))
_two = lambda: [dict(heading='h7', colophon='c8', verses=[V(1, 'a'), V(2, 'b'), V(25, 'y', 'p0107a'), V(2, 'z', 'p0108a'), V(3, 'w', 'p0108a')])]
_p = []; _c = fix_chapters(_two(), [(None, 2, 'start', None, 'p0108a', 'the leaf before it lost')], _p)   # two verses 2 in one chapter: the page picks
test_eq([[v['n'] for v in c['verses']] for c in _c], [[1, 2, 25], [2, 3]]); test_eq(_p, ['p0108a: a chapter starts at verse 2 (the leaf before it lost)'])
test_eq([[v['n'] for v in c['verses']] for c in fix_chapters(_two(), [(1, 2, 'start', None, 2, '')])], [[1, 2, 25], [2, 3]])   # or which one, from 1
test_eq([[v['n'] for v in c['verses']] for c in fix_chapters(_two(), [(1, 2, 'start', None, None, '')])], [[1], [2, 25, 2, 3]])   # else the first
_p = []; fix_chapters(_two(), [(1, 2, 'start', None, 'p0999a', '')], _p); test_eq(_p, ['1.2: no verse 2 after the first of a chapter (p0999a); the start not applied'])
_p = []
test_eq(sutras([dict(verses=[V(1, 'अथातो धर्म-\nजिज्ञासा ।'), V(2, 'चोदनालक्षणोऽर्थो धर्मः ॥')])], _p)[0]['verses'][0]['text'], 'अथातो धर्मजिज्ञासा ॥'); test_eq(_p, ['1.1 (1): closing ॥ supplied'])

In [ ]:
#| hide
import nbdev; nbdev.nbdev_export()